In [0]:


# =====================================
# IMPORT LIBRARIES
# =====================================

import dlt
from pyspark.sql.functions import col


# =====================================
# CREATE VIEW
# =====================================
@dlt.view(name="sales_view")
def sales_view():
    df = spark.readStream.table("sales")
    return df.withColumn("total", col("amount") * col("quantity"))

# =====================================
# CREATE TARGET TABLE
# =====================================


dlt.create_streaming_table(
    name="sales_transformed",
    comment="sales data with total amount"
)


### Store all changes in a table as scd type 2 using Databricks Auto CDC

# =====================================
# APPLY CDC (SCD TYPE 2)
# =====================================


dlt.create_auto_cdc_flow(
    source="sales_view",
    target="sales_transformed",
    keys=["sales_id"],
    sequence_by="sale_timestamp",
    ignore_null_updates=False,
    except_column_list=[],
    stored_as_scd_type="2"
)
